# Generador v3, paso 1: el profesor escribe las respuestas con causas del pasaje (Colab T4)

**Qué cambia frente al v2.** Los casos son los mismos, pero:
1. **Pasajes de la app** (`casos_v3.jsonl`): cada examen alto o bajo trae el fragmento de MedlinePlus que explica ese
   resultado, el mismo que recibe el generador en la app. En el v1 y el v2, el 92 % de los exámenes alterados de
   train tenía otro fragmento, casi siempre genérico ("la prueba se incluye en un chequeo de rutina").
2. **Guía nueva** (`GUIA_PROFESOR_V3`): las causas se toman del pasaje; si el pasaje no trae causas, no se inventan.
   Antes los profesores escribían causas de memoria y el 46-56 % no estaba en el pasaje; el v1 y el v2 subieron a
   59-65 %.
3. **Pasaje completo** (700 caracteres, como el estudiante). Antes se cortaba a 350 por las cuotas de Groq.
4. **3 respuestas por caso**: `dataset.py --version v3` se queda con la mejor que pase todos los filtros.

Profesor: Qwen2.5-14B-Instruct-AWQ con vLLM (sin cuotas). Casos: 1.000 de train y 100 de val.

**Tiempo:** unas 3 a 4 horas. Se guarda en Drive después de cada lote de 64 casos: si Colab se desconecta, vuelve a
correr las celdas 1 a 3 y luego la 4, y sigue donde iba. Puedes pararlo antes: con unos 600 casos ya alcanza.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/letrademedico'
!mkdir -p {DRIVE}/respuestas_v3

In [ ]:
%cd /content
!test -d Proyecto_LaLetraDeMedico || git clone https://github.com/Valero25/Proyecto_LaLetraDeMedico.git
%cd /content/Proyecto_LaLetraDeMedico
!git pull -q && git log --oneline -1
# Colab usa Python 3.13: vLLM sin versión fija (las versiones viejas no existen para 3.13)
!pip install -q vllm
!wc -l data/generador/casos_v3.jsonl

## 1. Generar (reanudable)

In [ ]:
%cd /content/Proyecto_LaLetraDeMedico
!python training/generador/profesor_local.py --version v3 --muestras 3 --salida {DRIVE}/respuestas_v3

In [ ]:
# Cuántos casos van y cuántas respuestas pasan las verificaciones básicas
import json, collections
filas = [json.loads(l) for l in open(f'{DRIVE}/respuestas_v3/qwen2.5-14b-instruct-awq.jsonl', encoding='utf-8')]
casos = {f['id'] for f in filas}
limpias = sum(bool(f['verificacion'].get('json_valido')) and f['verificacion']['cobertura'] == 1 for f in filas)
print(len(casos), 'casos,', len(filas), 'respuestas,', limpias, 'con JSON válido y todos los exámenes')
print(collections.Counter(f['split'] for f in filas))

## 2. Descargar las respuestas (unos 10 MB)

In [ ]:
!cd {DRIVE} && zip -qr /content/respuestas_v3.zip respuestas_v3 && ls -lh /content/respuestas_v3.zip
from google.colab import files
files.download('/content/respuestas_v3.zip')

## 3. En el computador local

1. Descomprime `respuestas_v3.zip` dentro de `data/generador/`. Debe quedar
   `data/generador/respuestas_v3/qwen2.5-14b-instruct-awq.jsonl`.
2. Avísale a Claude: arma el dataset (`python training/generador/dataset.py --version v3`), lo revisa y lo subes con
   `git push` para el paso 2 (`qlora_v3_colab.ipynb`).